In [1]:
import pathlib as pl
from platform import processor
from pprint import pprint
from shutil import rmtree
from sys import platform
import warnings

import pydoc

import hvplot.pandas  # noqa
import jupyter_black
import numpy as np
import pywatershed as pws
from pywatershed.utils import gis_files
from tqdm import tqdm
import xarray as xr

gis_files.download()  # this downloads GIS files

jupyter_black.load()  # auto-format the code in this notebook

prms_channel_flow_graph jit compiling with numba


In [2]:
domain_dir = pws.constants.__pywatershed_root__ / "data/drb_2yr"
nb_output_dir = pl.Path("./02_prms_legacy_models_monthly_output")

In [3]:
params = pws.parameters.PrmsParameters.load(domain_dir / "myparam.param")

In [4]:
nhm_processes = [
    pws.PRMSSolarGeometry,
    pws.PRMSAtmosphere,
    pws.PRMSCanopy,
    pws.PRMSSnow,
    pws.PRMSRunoff,
    pws.PRMSSoilzone,
    pws.PRMSGroundwater,
    pws.PRMSChannel,
]

In [5]:
control = pws.Control.load_prms(
    domain_dir / "nhm.control", warn_unused_options=False
)

In [6]:
cbh_nc_dir = domain_dir
control.options["netcdf_output_var_names"] += ["infil_hru", "sroff_vol"]
control.edit_end_time(np.datetime64("1979-07-01T00:00:00"))
run_dir = nb_output_dir / "nhm"
if run_dir.exists():
    rmtree(run_dir)
control.options = control.options | {
    "input_dir": cbh_nc_dir,
    "budget_type": "warn",
    "calc_method": "numba",
    "netcdf_output_dir": run_dir,
}

In [7]:
nhm = pws.Model(
    nhm_processes,
    control=control,
    parameters=params,
)

PRMSCanopy jit compiling with numba 
PRMSSnow jit compiling with numba 
PRMSRunoff jit compiling with numba 
PRMSSoilzone jit compiling with numba 
PRMSGroundwater jit compiling with numba 
PRMSChannel jit compiling with numba 


In [8]:
output = pws.base.CustomOutput(
    control=control,
    model=nhm,
    monthly_accum_var_list=[
        "sroff",
        "hru_actet",
    ],
    monthly_accum_stats=["accum", "mean"],
)

In [9]:
for tt in tqdm(range(control.n_times)):
    nhm.advance()
    nhm.calculate()
    output.calculate()

nhm.finalize()
output.finalize()

  0%|                                                   | 0/182 [00:00<?, ?it/s]

model initializing NetCDF output


100%|█████████████████████████████████████████| 182/182 [00:06<00:00, 28.56it/s]


In [10]:
control.itime_step

181

In [11]:
output.monthly_accumulations

{'sroff': <xarray.DataArray (month: 7, nhru: 765)> Size: 43kB
 array([[1.41290240e+00, 1.77532983e+00, 1.26035110e+00, ...,
         1.24604140e-03, 1.03197458e-02, 5.08721212e-04],
        [2.10963073e+00, 2.34014965e+00, 1.92590646e+00, ...,
         4.79010914e-04, 4.58618068e-03, 0.00000000e+00],
        [2.21787952e+00, 2.57662505e+00, 1.37259150e+00, ...,
         3.10495592e-03, 5.23775888e-01, 0.00000000e+00],
        ...,
        [1.17656129e+00, 2.14347113e+00, 2.52574224e-01, ...,
         9.38922069e-01, 1.07867749e+00, 0.00000000e+00],
        [1.36060380e+00, 1.80460074e+00, 2.84524369e-01, ...,
         2.28762199e-01, 1.15906431e-01, 0.00000000e+00],
        [3.14475006e-02, 1.74638776e-01, 1.55929690e-02, ...,
         1.85587754e-02, 4.62672147e-03, 0.00000000e+00]], shape=(7, 765))
 Coordinates:
   * month    (month) datetime64[s] 56B 1979-01-01 1979-02-01 ... 1979-07-01
   * nhru     (nhru) int64 6kB 5307 5308 5309 5310 5311 ... 7248 7249 7250 7251
 Attributes:
    

In [12]:
output.n_days_per_month

array([31, 28, 31, 30, 31, 30,  1], dtype=int32)